In [ ]:
#| default_exp machine_learning.training.token_classification

> **AI-generated — pending human review.** Created by Claude Code on 2026-10-22. Review checklist: `_nbdev_development/AI_REVIEW_CHECKLIST.md` (workspace root). Remove this cell only after review passes.

# Token classification training (zero-width seams)

Consumes the row contract of `machine_learning.producers.note_division`: each row is one text with zero-width `{start, end, label: 'split'}` spans. Nothing here knows about vaults, datasets on disk, or machines — data paths and devices are caller-supplied, so this module is safe to publish.

**Label rule.** Byte-level BPE means token boundaries do not line up with words, so labels are snapped by *character offsets*: the last non-blank token whose range **ends at** a seam gets label 1; a whitespace-only token covering the seam (the `\n` joiner in canonical rows) gets `-100` (ignored); everything else, including padding and special tokens, is ignored or 0. Train on the newline-stripped view (`ml_augmentation.division_strip_newlines_v1`) so the seam carries no input-visible cue; truncation cuts spans past `max_length` out of the window — such rows lose their positive and act as negatives, which is why streams are pair-bounded.

In [ ]:
#| export
from os import PathLike
from pathlib import Path

import contextlib
import os
import time
import torch
from transformers import (AutoModelForTokenClassification,
                                      AutoTokenizer)


## Labels: zero-width seams → per-token `0/1/-100`

In [ ]:
#| export
def token_labels_for_spans(text: str, spans: list[dict], offsets: list[tuple]) -> list[int]:
    """
    Per-token `0/1/-100` labels: 1 on the last non-blank token ending exactly at a seam.

    `offsets` are the tokenizer's character `offset_mapping` for the same text
    (special tokens excluded, i.e. `add_special_tokens=False`). A whitespace-only
    token *starting at* a seam (the canonical `\n` joiner) is `-100` — invisible
    to the loss; in the newline-stripped view no such token exists and the seam
    is an ordinary space. Empty ranges (special tokens) never match.

    **Parameters**
    - `text` (*str*) — The row text.
    - `spans` (*list[dict]*) — Zero-width spans with `label == 'split'`.
    - `offsets` (*list[tuple]*) — `(start, end)` char range per token.
    """
    seams = {s['start'] for s in spans if s.get('label') == 'split'}
    labels = []
    for b, e in offsets:
        if e > b and e in seams and text[b:e].strip():
            labels.append(1)
        elif b in seams and not text[b:e].strip():
            labels.append(-100)
        else:
            labels.append(0)
    return labels


In [ ]:
from fastcore.test import test_eq

# canonical view: 'ab' ends at the seam -> 1; the '\n' token covering it -> -100.
offs = [(0, 2), (2, 3), (3, 5)]
seams = [{'start': 2, 'end': 2, 'label': 'split'}]
test_eq(token_labels_for_spans('ab\ncd', seams, offs), [1, -100, 0])
# stripped view: seam glued to the next token -> no -100 anywhere, just 1 then 0.
test_eq(token_labels_for_spans('ab cd', seams, [(0, 2), (2, 5)]), [1, 0])
# negatives: no spans -> all zero; blank token touching a seam start is not positive.
test_eq(token_labels_for_spans('zz yy', [], [(0, 2), (2, 3), (3, 5)]), [0, 0, 0])


## Features and scoring

In [ ]:
#| export
def division_feature_tensors(rows: list[dict], tokenizer, max_length: int = 512) -> dict:
    """
    Rows → padded `input_ids`/`attention_mask`/`labels` tensors (`-100` pad on labels).

    No special tokens (offsets stay aligned with plain text). Truncation past
    `max_length` drops seams whose token vanished — the row silently becomes all-
    negative; division rows are pair-bounded so this is rare, but check
    `(labels == 1).sum()` against your seam count when changing `max_length`.

    **Parameters**
    - `rows` (*list[dict]*) — Rows with `text` and zero-width `spans`.
    - `tokenizer` — A fast tokenizer (needs `offset_mapping`).
    - `max_length` (*int*) — Truncation/padding width cap.
    """
    texts = [r['text'] for r in rows]
    enc = tokenizer(texts, return_offsets_mapping=True, add_special_tokens=False,
                    truncation=True, max_length=max_length, padding="longest")
    labels = [token_labels_for_spans(t, r['spans'], o)
              for t, r, o in zip(texts, rows, enc.pop('offset_mapping'))]
    width = len(enc['input_ids'][0])
    return {'input_ids': torch.tensor(enc['input_ids']),
            'attention_mask': torch.tensor(enc['attention_mask']),
            'labels': torch.tensor([l + [-100] * (width - len(l)) for l in labels])}


In [ ]:
#| export
def score_seam_f1(model, tokenizer, rows: list[dict], max_length: int = 512,
                  batch_size: int = 8) -> dict:
    """
    Seam-level precision/recall/F1: predicted positives are label-1 tokens; a hit
    is a prediction whose end offset equals a gold seam's `start`.

    Exact-boundary matching (zero-width spans), micro-averaged over rows — the
    gate metric for the division task. Runs on the model's current device.

    **Parameters**
    - `model` — A loaded `AutoModelForTokenClassification` (`num_labels=2`).
    - `tokenizer` — The model's tokenizer.
    - `rows` (*list[dict]*) — Scored rows (pure eval rows by convention).
    - `max_length` (*int*) — Must match training or seams truncate away.
    - `batch_size` (*int*) — Inference batch size.
    """
    model.eval()
    tp = fp = fn = 0
    for i in range(0, len(rows), batch_size):
        chunk = rows[i:i + batch_size]
        feats = division_feature_tensors(chunk, tokenizer, max_length)
        with torch.no_grad(), _autocast(model.device):
            preds = model(input_ids=feats['input_ids'].to(model.device),
                          attention_mask=feats['attention_mask'].to(model.device)
                          ).logits.argmax(-1)
        offs = tokenizer([r['text'] for r in chunk], add_special_tokens=False,
                         truncation=True, max_length=max_length,
                         return_offsets_mapping=True)['offset_mapping']
        for r, pred_row, o in zip(chunk, preds.tolist(), offs):
            gold = {s['start'] for s in r['spans'] if s['label'] == 'split'}
            pred = {o[k][1] for k in range(len(o)) if pred_row[k] == 1 and o[k][1] > o[k][0]}
            tp += len(pred & gold); fp += len(pred - gold); fn += len(gold - pred)
    p = tp / (tp + fp) if tp + fp else 0.0
    rc = tp / (tp + fn) if tp + fn else 0.0
    return {'precision': p, 'recall': rc, 'f1': 2 * p * rc / (p + rc) if p + rc else 0.0}


## Training loop (plain PyTorch — no HF Trainer dependency)

In [ ]:
#| export
def _pick_device(device: str = None):
    """`device` string or the best available: cuda → mps → cpu."""
    if device:
        return torch.device(device)
    if torch.cuda.is_available():
        return torch.device('cuda')
    if getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


def _batch_order(rows: list[dict], batch_size: int) -> list[list[int]]:
    """Row indices as batches grouped by similar text length (char-count proxy).

    Similar-length rows keep per-batch padding tiny and, on MPS, cut the number
    of distinct tensor shapes the caching allocator must keep segments for.
    """
    order = sorted(range(len(rows)), key=lambda i: len(rows[i]['text']))
    return [order[i:i + batch_size] for i in range(0, len(rows), batch_size)]


def _autocast(device):
    """bf16 autocast context on cuda/mps; a no-op context elsewhere."""
    if device.type in ('cuda', 'mps'):
        return torch.autocast(device_type=device.type, dtype=torch.bfloat16)
    return contextlib.nullcontext()


def _progress(msg: str) -> None:
    """Append a timestamped `msg` line to `$TROUVER_PROGRESS_FILE`; no-op if unset."""
    path = os.environ.get('TROUVER_PROGRESS_FILE')
    if path:
        with open(path, 'a') as f:
            f.write(f"{time.strftime('%H:%M:%S')} {msg}\n")


def _eta(elapsed: float, step: int, nb: int, epochs: int, ep: int) -> str:
    """`'epoch_eta Xm total_eta Ym'` from the mean step speed so far."""
    sps = step / max(elapsed, 1e-9)
    return (f'epoch_eta {(nb - step) / sps / 60:.0f}m '
            f'total_eta {((epochs - ep) * nb + nb - step) / sps / 60:.0f}m')


def _epoch_eval_loss(model, rows: list[dict], tokenizer, device, max_length: int,
                     batch_size: int, cap: int = 256) -> float:
    """Mean loss over up to `cap` eval rows (no grad + autocast; train mode restored)."""
    model.eval()
    tot = n = 0
    with torch.no_grad():
        for i in range(0, min(cap, len(rows)), batch_size):
            mb = {k: v.to(device) for k, v in
                  division_feature_tensors(rows[i:i + batch_size], tokenizer,
                                           max_length).items()}
            with _autocast(device):
                tot += float(model(**mb).loss)
            n += 1
    model.train()
    return tot / max(n, 1)


def _fine_tune(model, rows: list[dict], tokenizer, device, epochs: int,
               batch_size: int, lr: float, max_length: int,
               eval_rows: list[dict] = None, out_dir: PathLike = None) -> None:
    """AdamW over length-bucketed batches; bf16 autocast off-CPU + MPS cache trims.

    Batch order is reshuffled per epoch from the global RNG (seeded by
    `train_token_classifier`). Variable-length rows fragment the MPS pool until
    it OOMs, so every 50 steps on MPS we hand cached segments back.

    Progress sidecar (`$TROUVER_PROGRESS_FILE`, no-op when unset): a settings
    line at start, one line per 50 steps with loss + eta clocks, and an
    `eval_loss` line after each epoch when `eval_rows` is given. With
    `out_dir`, model + tokenizer are also checkpointed to
    `out_dir/checkpoints/epoch-N` at every epoch boundary (crash insurance).
    """
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()
    _progress(f'start rows={len(rows)} epochs={epochs} batch_size={batch_size} '
              f'max_length={max_length} lr={lr} device={device.type}')
    t0 = time.monotonic()
    for ep in range(1, epochs + 1):
        batches = _batch_order(rows, batch_size)
        for step, bi in enumerate(torch.randperm(len(batches)).tolist(), 1):
            mb = division_feature_tensors([rows[k] for k in batches[bi]],
                                          tokenizer, max_length)
            mb = {k: v.to(device) for k, v in mb.items()}
            with _autocast(device):
                loss = model(**mb).loss
            opt.zero_grad(); loss.backward(); opt.step()
            if device.type == 'mps' and step % 50 == 0:
                torch.mps.empty_cache()
            if step % 50 == 0:
                _progress(f'epoch {ep}/{epochs} step {step}/{len(batches)} '
                          f'loss {loss.item():.4f} '
                          + _eta(time.monotonic() - t0, step, len(batches), epochs, ep))
        if eval_rows:
            _progress(f'epoch {ep}/{epochs} eval_loss '
                      f'{_epoch_eval_loss(model, eval_rows, tokenizer, device, max_length, batch_size):.4f}')
        if out_dir: _save_checkpoint(model, tokenizer, out_dir, ep)


def _save_checkpoint(model, tokenizer, out_dir: PathLike, epoch: int) -> None:
    """Save model + tokenizer to `out_dir/checkpoints/epoch-N` and log a line."""
    d = Path(out_dir) / 'checkpoints' / f'epoch-{epoch}'
    model.save_pretrained(d); tokenizer.save_pretrained(d)
    _progress(f'saved checkpoint {d}')


def train_token_classifier(rows: list[dict], model_id: str, out_dir: PathLike,
                           eval_rows: list[dict] = None, epochs: int = 3,
                           batch_size: int = 8, lr: float = 2e-5, max_length: int = 512,
                           device: str = None, seed: int = 0,
                           gradient_checkpointing: bool = False) -> dict:
    """
    Fine-tune `model_id` + 2-class head on division rows; save to `out_dir`; score eval.

    Labels are `{0: 'none', 1: 'split'}`. Rows are grouped into length-bucketed
    batches padded to their own longest row and trained under bf16 autocast
    off-CPU (with periodic MPS cache trims) — variable-length rows otherwise
    fragment the MPS pool until it OOMs well below the machine's real memory.
    Returns `score_seam_f1` on `eval_rows` (pure, by convention) or `None`.

    **Parameters**
    - `rows` (*list[dict]*) — Training rows (augmented view recommended; see notebook text).
    - `model_id` (*str*) — Encoder id, e.g. from `model_registry.get_model_id`.
    - `out_dir` (*PathLike*) — Directory for the saved HF model + tokenizer;
      per-epoch crash checkpoints land under its `checkpoints/` subdir.
    - `gradient_checkpointing` (*bool*) — Recompute activations in backward on
      cuda/mps (big memory saving, ~25-35% speed cost); no-op on cpu.
    - `eval_rows` (*list[dict]*) — Optional rows scored after training (leakage-free
      by caller); also logged as epoch-end `eval_loss` in the progress sidecar.
    - `epochs` (*int*) / `batch_size` (*int*) / `lr` (*float*) — Standard knobs.
    - `max_length` (*int*) — Window cap; must match `score_seam_f1` at eval time.
    - `device` (*str*) — Force a device (`'cpu'`, `'mps'`, `'cuda'`); default = best available.
    - `seed` (*int*) — Torch seed for run reproducibility.
    """
    torch.manual_seed(seed)
    device = _pick_device(device)
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForTokenClassification.from_pretrained(
        model_id, num_labels=2, id2label={0: 'none', 1: 'split'},
        label2id={'none': 0, 'split': 1}).to(device)
    if gradient_checkpointing and device.type in ('cuda', 'mps'):
        model.gradient_checkpointing_enable()
    _fine_tune(model, rows, tokenizer, device, epochs, batch_size, lr,
               max_length, eval_rows, out_dir)
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    model.save_pretrained(out_dir); tokenizer.save_pretrained(out_dir)
    return None if eval_rows is None else score_seam_f1(
        model, tokenizer, eval_rows, max_length=max_length, batch_size=batch_size)


In [ ]:
#| hide
import tempfile
from pathlib import Path

from fastcore.test import *

from trouver.helper.tests import _test_directory
from trouver.machine_learning.ml_data import PROVENANCE_COLUMN, load_rows
from trouver.machine_learning.producers.note_division import NOTE_DIVISION_SCHEMA

FIXTURE = _test_directory() / 'ml' / 'note_division_demo.jsonl'
TINY = 'hf-internal-testing/tiny-random-BertForTokenClassification'


In [ ]:
# End-to-end on the synthetic fixture with a tiny random model: shapes, save, reload,
# and metric structure (tiny random weights — only structure is asserted, per spec).
rows = load_rows(FIXTURE, schema=NOTE_DIVISION_SCHEMA)
assert len(rows) == 3 and all(r[PROVENANCE_COLUMN].startswith("_index_demo_") for r in rows)
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(TINY)
feats = division_feature_tensors([dict(r) for r in rows], tok)
assert feats['input_ids'].shape == feats['labels'].shape
assert int((feats['labels'] == 1).sum()) == 3  # one seam per row survived alignment
with tempfile.TemporaryDirectory() as td:
    res = train_token_classifier([dict(r) for r in rows], TINY, td,
                                 eval_rows=[dict(r) for r in rows], epochs=1,
                                 batch_size=2, max_length=64, device='cpu')
    assert (Path(td) / 'config.json').exists()
    assert set(res) == {'precision', 'recall', 'f1'}
    assert all(0.0 <= v <= 1.0 for v in res.values())


In [ ]:
# length-bucketing + autocast helpers (MPS memory fix)
bs_ = _batch_order(rows, 3)
test_eq(sum(len(b) for b in bs_), len(rows))           # every row kept exactly once
test_eq(sorted(i for b in bs_ for i in b), list(range(len(rows))))
for b in bs_:                                          # each batch sorted by text length
    ls = [len(rows[i]['text']) for i in b]
    test_eq(ls, sorted(ls))
test_eq(len(bs_), -(-len(rows) // 3))                  # ceil division
import contextlib
assert isinstance(_autocast(torch.device('cpu')), contextlib.nullcontext)
with _autocast(torch.device('cpu')):                   # usable as a context manager
    pass

In [ ]:
# _progress sidecar logging ($TROUVER_PROGRESS_FILE)
import os as _os, tempfile as _tf
_p = Path(_tf.mkdtemp()) / 'prog.log'
_os.environ['TROUVER_PROGRESS_FILE'] = str(_p)
_progress('hello'); _progress('world')
lines = _p.read_text().splitlines()
test_eq([l.split(' ', 1)[1] for l in lines], ['hello', 'world'])  # ts prefix + msg
del _os.environ['TROUVER_PROGRESS_FILE']
_progress('dropped')                                              # no-op when unset
test_eq(len(_p.read_text().splitlines()), 2)

In [ ]:
# eta clocks + epoch eval loss (progress v2)
test_eq(_eta(100.0, 100, 1000, 3, 1), 'epoch_eta 15m total_eta 48m')
test_eq(_eta(50.0, 100, 100, 1, 1), 'epoch_eta 0m total_eta 0m')   # last batch done
_m = AutoModelForTokenClassification.from_pretrained(TINY, num_labels=2)
_v = _epoch_eval_loss(_m, [dict(r) for r in rows], tok, torch.device('cpu'), 64, 2)
assert isinstance(_v, float) and _v > 0                             # tiny random model: finite positive mean
test_eq(_epoch_eval_loss(_m, [], tok, torch.device('cpu'), 64, 2), 0.0)

In [ ]:
# per-epoch crash checkpoints + gradient-checkpointing flag (v3)
import tempfile
with tempfile.TemporaryDirectory() as td_:
    _save_checkpoint(_m, tok, td_, 1)
    assert (Path(td_) / 'checkpoints' / 'epoch-1' / 'config.json').exists()
    res_ = train_token_classifier([dict(r) for r in rows], TINY, td_,
                                  eval_rows=[dict(r) for r in rows], epochs=1,
                                  batch_size=2, max_length=64, device='cpu',
                                  gradient_checkpointing=True)   # no-op on cpu path
    assert set(res_) == {'precision', 'recall', 'f1'}
    assert (Path(td_) / 'checkpoints' / 'epoch-1').exists()      # epoch save ran